In [1]:
import pandas as pd
from datetime import datetime, timedelta
from massive import RESTClient

API_KEY = 	'geI0LniQ6cMXjO7WftfHwwaUr5XpEL_T'
client = RESTClient(API_KEY)

In [2]:
two_years = datetime.now() - timedelta(days=730)
two_years.strftime('%Y-%m-%d')

'2024-02-12'

In [3]:
def data_collection(tickers):
    data = pd.DataFrame()
    
    two_years = datetime.now() - timedelta(days=730)
    two_years = two_years.strftime('%Y-%m-%d')
    today = datetime.now()
    today = today.strftime('%Y-%m-%d')

    for ticker in tickers:
        aggs=[]
        for a in client.list_aggs(ticker=ticker, multiplier=1, timespan="day", from_=two_years, to=today):
            aggs.append(a)
            df = pd.DataFrame(aggs)
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
            df['ticker'] = ticker
        data = pd.concat([data, df])
        
    return data

In [5]:
tickers = ['SPY', 'VT', 'BND', 'VNLA', 'VNQ']
 
new_data = data_collection(tickers)

In [6]:
new_data

,open,high,low,close,volume,vwap,timestamp,transactions,otc,ticker
0,501.17,503.50,500.240,500.98,56502283.0,501.5388,2024-02-12 05:00:00+00:00,437189,None,SPY
1,494.53,497.09,490.715,494.08,113099199.0,494.2840,2024-02-13 05:00:00+00:00,779480,None,SPY
2,496.79,499.07,494.400,498.57,68387827.0,496.7195,2024-02-14 05:00:00+00:00,536843,None,SPY
3,499.29,502.20,498.795,502.01,61682960.0,500.7639,2024-02-15 05:00:00+00:00,516093,None,SPY
4,501.70,502.87,498.750,499.51,75532928.0,500.9231,2024-02-16 05:00:00+00:00,532752,None,SPY
...,...,...,...,...,...,...,...,...,...,...
496,90.24,91.32,89.890,90.95,3941137.0,90.8873,2026-02-04 05:00:00+00:00,54122,None,VNQ
497,90.76,91.33,90.240,90.82,3588497.0,90.8332,2026-02-05 05:00:00+00:00,63752,None,VNQ
498,91.53,92.38,91.325,92.25,3638714.0,91.8650,2026-02-06 05:00:00+00:00,51752,None,VNQ
499,92.16,92.72,91.600,92.64,3021088.0,92.3873,2026-02-09 05:00:00+00:00,43033,None,VNQ


In [7]:
old_data = pd.read_csv('historical_data.csv').drop(columns=['Unnamed: 0'])
old_data['timestamp'] = pd.to_datetime(old_data['timestamp'])
old_data

,open,high,low,close,volume,vwap,timestamp,transactions,otc,ticker
0,472.53,474.9200,472.450,474.84,55761805.0,473.9757,2023-12-19 05:00:00+00:00,416103,NaN,SPY
1,473.96,475.8950,467.820,468.26,102920959.0,471.6749,2023-12-20 05:00:00+00:00,666863,NaN,SPY
2,471.33,472.9750,468.840,472.70,86667465.0,471.2282,2023-12-21 05:00:00+00:00,600897,NaN,SPY
3,473.86,475.3800,471.700,473.65,67160419.0,473.7998,2023-12-22 05:00:00+00:00,486178,NaN,SPY
4,474.07,476.5800,473.990,475.65,55386952.0,475.1113,2023-12-26 05:00:00+00:00,348986,NaN,SPY
...,...,...,...,...,...,...,...,...,...,...
2500,89.43,89.9200,89.260,89.56,5199634.0,89.5347,2025-12-11 05:00:00+00:00,49913,NaN,VNQ
2501,89.95,90.2700,89.230,89.45,3844045.0,89.5681,2025-12-12 05:00:00+00:00,47659,NaN,VNQ
2502,89.82,89.8800,89.095,89.73,3895890.0,89.5061,2025-12-15 05:00:00+00:00,52276,NaN,VNQ
2503,89.65,89.9799,88.925,89.07,3698731.0,89.3127,2025-12-16 05:00:00+00:00,54438,NaN,VNQ


In [8]:
combined_df = pd.concat([old_data, new_data], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=['timestamp', 'ticker'], keep='last')
combined_df = combined_df.sort_values(['timestamp', 'ticker']).reset_index(drop=True)
combined_df

,open,high,low,close,volume,vwap,timestamp,transactions,otc,ticker
0,73.36,73.470,73.3400,73.39,6924239.0,73.3976,2023-12-19 05:00:00+00:00,27494,NaN,BND
1,472.53,474.920,472.4500,474.84,55761805.0,473.9757,2023-12-19 05:00:00+00:00,416103,NaN,SPY
2,48.35,48.380,48.3500,48.35,725110.0,48.3622,2023-12-19 05:00:00+00:00,719,NaN,VNLA
3,88.50,88.996,88.4200,88.75,5787080.0,88.7594,2023-12-19 05:00:00+00:00,53518,NaN,VNQ
4,101.67,102.190,101.6700,102.17,1992131.0,102.0314,2023-12-19 05:00:00+00:00,12852,NaN,VT
...,...,...,...,...,...,...,...,...,...,...
2680,74.45,74.520,74.4300,74.47,8288363.0,74.4730,2026-02-10 05:00:00+00:00,36639,None,BND
2681,694.95,696.540,691.6600,692.12,65178906.0,694.0032,2026-02-10 05:00:00+00:00,997647,None,SPY
2682,49.24,49.250,49.2335,49.25,276473.0,49.2453,2026-02-10 05:00:00+00:00,937,None,VNLA
2683,92.76,94.110,92.7400,93.88,4265721.0,93.6655,2026-02-10 05:00:00+00:00,52115,None,VNQ


In [9]:
#data.to_csv('data.csv')
combined_df.to_csv('data.csv')